In [6]:
import sys, os, time
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
import numpy as np
import jax
import jax.numpy as jnp
import torch
import h5py

sys.path.insert(0, '../jaxtpc')
sys.path.insert(0, '..')
from tools.geometry import generate_detector
from tools.simulation import DetectorSimulator
from tools.loader import load_event

from goop import OpticalSimConfig, OpticalSimulator, write_config_light, save_event_light
from goop.kernels import SERKernel
from goop.delays import ScintillationBiexponentialDelay, TTSDelay, TPBTriexponentialDelay
from goop.digitize import DigitizationConfig
from goop.sampler import create_default_tof_sampler

In [12]:
DATA_PATH = '/sdf/home/y/youngsam/sw/dune/sirentv/data/out.h5'
CONFIG_PATH = '../jaxtpc/config/cubic_wireplane_config.yaml'
OUTPUT_PATH = '../data/light_output_interaction.h5'
N_SAVE = 100
TOTAL_PAD = 250_000
RESPONSE_CHUNK_SIZE = 50_000
SAVE_INTERACTIONS = True

N_BITS = 15
ADC_MAX = (1 << N_BITS) - 1
PEDESTAL = 0.9 * ADC_MAX
MAX_PE_PER_PMT = 90_000

In [8]:
# jaxtpc detector + simulator
detector_config = generate_detector(CONFIG_PATH)
sim = DetectorSimulator(detector_config, total_pad=TOTAL_PAD,
                        response_chunk_size=RESPONSE_CHUNK_SIZE, include_track_hits=False)
cfg = sim.config

# JIT warm-up
deposits = load_event(DATA_PATH, cfg, event_idx=0)
filled = sim.process_event_light(deposits)
jax.block_until_ready(filled.volumes[0].charge)
print(f'JIT done. {cfg.n_volumes} volume(s).')

# GOOP simulator
goop_config = OpticalSimConfig(
    tof_sampler=create_default_tof_sampler(lazy=False),
    delays=[ScintillationBiexponentialDelay(singlet_fraction=0.3, tau_singlet_ns=6.0, tau_triplet_ns=1300.0),
            TPBTriexponentialDelay(), TTSDelay(fwhm_ns=2.4, apply_transit_time=True)],
    tick_ns=1.0,
    kernel=SERKernel(device=torch.device('cuda'), duration_ns=10_000),
    gain=2**N_BITS / MAX_PE_PER_PMT,
    oversample=10,
    ser_jitter_std=0.1,
    baseline_noise_std=2.6,
    digitization=DigitizationConfig(n_bits=N_BITS, pedestal=PEDESTAL),
)
goop_sim = OpticalSimulator(goop_config)
print(f'GOOP ready: {N_BITS}-bit ADC, pedestal={PEDESTAL:.0f}, {goop_config.n_channels} channels')

--- Creating DetectorSimulator ---
   Extracting parameters...


E0402 11:39:02.564860 3264201 numa_hwloc.cc:121] Call to hwloc_set_cpubind() failed: Invalid argument [22]
E0402 11:39:02.564968 3264202 numa_hwloc.cc:121] Call to hwloc_set_cpubind() failed: Invalid argument [22]
E0402 11:39:02.565150 3264203 numa_hwloc.cc:121] Call to hwloc_set_cpubind() failed: Invalid argument [22]


   Loading response kernels...
   Space charge effects: DISABLED
   Recombination model: emb
   Config: total_pad=250,000, response_chunk=50,000, num_s=16, K_wire=2, K_time=6
   Track labeling: DISABLED
   Volumes: 2
--- DetectorSimulator Ready ---
JIT done. 2 volume(s).
GOOP ready: 15-bit ADC, pedestal=29490, 162 channels


In [ ]:
with h5py.File(OUTPUT_PATH, 'w') as f:
    write_config_light(f, goop_config, label_key=("volume", "interaction")[SAVE_INTERACTIONS], n_labels=cfg.n_volumes,
                       source_file=DATA_PATH, n_events=N_SAVE)

    print(f"{'Evt':>4} {'Segs':>8} {'Photons':>12} {'PEs':>10} {'Chunks':>7} {'t(s)':>6}")
    print('-' * 60)

    for evt in range(N_SAVE):
        deposits = load_event(DATA_PATH, cfg, event_idx=evt)
        res = sim.process_event_light(deposits)
        jax.block_until_ready(res.volumes[0].charge)

        all_pos, all_nph, all_t, all_labels = [], [], [], []
        for v in range(cfg.n_volumes):
            vol = res.volumes[v]
            n = vol.n_actual
            if n == 0:
                continue
            all_pos.append(vol.positions_mm[:n])
            all_nph.append(jnp.ceil(vol.photons[:n]).astype(jnp.int32))
            all_t.append(vol.t0_us[:n] * 1000.0)
            if SAVE_INTERACTIONS:
                all_labels.append(vol.interaction_ids[:n])
            else:
                all_labels.append(jnp.full((n,), v, dtype=jnp.int32))

        pos_mm = jnp.concatenate(all_pos)
        n_ph = jnp.concatenate(all_nph)
        t_ns = jnp.concatenate(all_t)
        labels = jnp.concatenate(all_labels)

        t0 = time.time()
        waveforms = goop_sim.simulate(pos_mm, n_ph, t_ns,
                                      labels=labels, stitched=True,
                                      subtract_t0=False, add_baseline_noise=False)
        elapsed = time.time() - t0

        save_event_light(f, f'event_{evt:03d}', waveforms,
                         source_event_idx=evt, digitized=True, n_bits=N_BITS)

        total_pe = sum(w.attrs['pe_counts'].sum().item() for w in waveforms)
        total_chunks = sum(w.n_chunks for w in waveforms)
        print(f'{evt:>4} {pos_mm.shape[0]:>8,} {int(n_ph.sum()):>12,} '
              f'{total_pe:>10,} {total_chunks:>7,} {elapsed:>5.2f}s')
        del waveforms

print('-' * 60)
print(f'Saved {N_SAVE} events to {OUTPUT_PATH}')

 Evt     Segs      Photons        PEs  Chunks   t(s)
------------------------------------------------------------
   0  226,796  371,059,868  1,659,872   2,524  1.87s
   1  263,503  407,046,307  1,987,738   2,695  1.80s
   2   94,886  137,520,911    686,312   2,177  1.30s
   3  209,010  313,502,764  1,400,430   2,524  1.70s
   4   98,513  181,166,077    910,811   1,838  1.09s
   5  220,704  395,567,305  1,930,375   2,857  1.97s
   6  151,408  220,567,945    978,192   2,176  1.50s
   7  171,890  304,734,259  1,492,110   2,668  1.67s
   8  150,203  223,211,791  1,158,231   2,500  1.53s
   9  178,142  313,496,343  1,530,332   2,002  1.26s
  10  360,100  598,299,716  2,845,339   3,678  2.53s
  11  234,300  391,362,169  1,678,655   3,517  2.39s
  12   89,508  150,089,086    697,674   2,173  1.38s
  13  141,998  231,739,587  1,001,158   2,340  1.41s
  14  240,501  354,046,099  1,871,229   1,991  1.33s
  15  258,454  450,765,559  2,103,439   3,197  2.12s
  16  208,069  323,135,039  1,586,620 

In [22]:
import os
from goop.io import load_event_light

fsize = os.path.getsize(OUTPUT_PATH)
print(f'File size: {fsize / 1e6:.1f} MB ({fsize / N_SAVE / 1e6:.1f} MB/event)')

with h5py.File(OUTPUT_PATH, 'r') as f:
    print(f'Config: label_key={f["config"].attrs["label_key"]}, n_labels={f["config"].attrs["n_labels"]}, '
          f'baseline_noise_std={f["config"].attrs["baseline_noise_std"]}')
    wfs = load_event_light(f, 'event_000', device='cpu')
    label_key = f['config'].attrs['label_key'].capitalize()
    for v, wf in enumerate(wfs):
        print(f'  {label_key} {v}: {wf.n_chunks} chunks, {wf.adc.numel():,} samples, '
              f'PE={wf.attrs["pe_counts"].sum().item():,}')

File size: 627.4 MB (6.3 MB/event)
Config: label_key=interaction, n_labels=2, baseline_noise_std=2.6
  Interaction 0: 162 chunks, 1,621,763 samples, PE=15
  Interaction 1: 171 chunks, 4,046,575 samples, PE=185,545
  Interaction 2: 170 chunks, 4,103,839 samples, PE=338,288
  Interaction 3: 165 chunks, 3,955,507 samples, PE=162,592
  Interaction 4: 171 chunks, 3,723,785 samples, PE=214,937
  Interaction 5: 173 chunks, 4,916,987 samples, PE=366,571
  Interaction 6: 164 chunks, 2,350,864 samples, PE=7,664
  Interaction 7: 165 chunks, 2,894,109 samples, PE=36,557
  Interaction 8: 172 chunks, 3,950,368 samples, PE=117,643
  Interaction 9: 164 chunks, 2,959,918 samples, PE=47,976
  Interaction 10: 167 chunks, 2,730,082 samples, PE=19,322
  Interaction 11: 167 chunks, 3,043,831 samples, PE=61,779
  Interaction 12: 168 chunks, 2,836,044 samples, PE=30,501
  Interaction 13: 170 chunks, 2,902,540 samples, PE=32,193
  Interaction 14: 175 chunks, 3,374,476 samples, PE=38,289
